In [ ]:
#Install dependencies
!pip install google-cloud-storage
!pip install pandas pyarrow

import pandas as pd
from google.cloud import storage
from io import BytesIO
import re # Import the re module for regular expressions

# ----------------------
# Configuration
# ----------------------
BUCKET_NAME = "dw-airbnb-data-lake"
BRONZE_PREFIX = "bronze_layer/raw_airbnb_pricing/"   # Raw Airbnb CSVs
SILVER_PARQUET_PATH = "silver_layer/pricing_table/airbnb_pricing_data.parquet"

# ----------------------
# 2. Authenticate GCP
# ----------------------
from google.colab import auth
auth.authenticate_user()
client = storage.Client()
bucket = client.bucket(BUCKET_NAME)

# ----------------------
# Load all Bronze Airbnb CSVs from GCS
# ----------------------
blobs = list(client.list_blobs(BUCKET_NAME, prefix=BRONZE_PREFIX))
csv_files = [b for b in blobs if b.name.endswith(".csv")]

dfs = []
for blob in csv_files:
    print(f"Loading {blob.name}")
    data = blob.download_as_bytes()
    df = pd.read_csv(BytesIO(data))

    # Extract city name from blob.name and add it as a column
    filename = blob.name.split('/')[-1]
    # Assuming filenames are like 'city_weekdays.csv' or 'city_weekends.csv'
    city_match = re.match(r'([a-z]+)_', filename, re.IGNORECASE)
    if city_match:
        city_name = city_match.group(1).capitalize() # Capitalize for consistency
        df['city'] = city_name
    else:
        df['city'] = 'Unknown' # Default if city cannot be extracted

    # Extract 'weekdays' or 'weekends' from filename and add as 'day_type' column
    day_type_match = re.search(r'_(weekdays|weekends)\.csv$', filename, re.IGNORECASE)
    if day_type_match:
        df['day_type'] = day_type_match.group(1).capitalize() # Capitalize for consistency
    else:
        df['day_type'] = 'Unknown' # Default if day type cannot be extracted

    dfs.append(df)

df_airbnb_pricing = pd.concat(dfs, ignore_index=True)
print("Combined Airbnb pricing data shape:", df_airbnb_pricing.shape)

# Rename 'Unnamed: 0' column to 'original_index' if it exists
if 'Unnamed: 0' in df_airbnb_pricing.columns:
    df_airbnb_pricing = df_airbnb_pricing.rename(columns={'Unnamed: 0': 'original_index'})

In [ ]:
display(airbnb_silver_layer.describe())

In [ ]:
# ----------------------
# Clean & normalize Airbnb data
# ----------------------
df_airbnb_pricing['guest_satisfaction_overall'] = pd.to_numeric(
    df_airbnb_pricing['guest_satisfaction_overall'], errors='coerce')
df_airbnb_pricing['attr_index_norm'] = pd.to_numeric(
    df_airbnb_pricing['attr_index_norm'], errors='coerce')

# Fill any NaNs introduced by to_numeric with the mean of the column
df_airbnb_pricing['guest_satisfaction_overall'] = df_airbnb_pricing['guest_satisfaction_overall'].fillna(df_airbnb_pricing['guest_satisfaction_overall'].mean())
df_airbnb_pricing['attr_index_norm'] = df_airbnb_pricing['attr_index_norm'].fillna(df_airbnb_pricing['attr_index_norm'].mean())

# Calculate realSum_norm
min_realSum = df_airbnb_pricing['realSum'].min()
max_realSum = df_airbnb_pricing['realSum'].max()
# Add a small epsilon to the denominator to prevent division by zero if max_realSum == min_realSum
denominator = (max_realSum - min_realSum)
if denominator == 0:
    # If all realSum values are the same, realSum_norm would be 0 or undefined. Handle this case.
    df_airbnb_pricing['realSum_norm'] = 0.0 # Or some other appropriate value
else:
    df_airbnb_pricing['realSum_norm'] = (
        df_airbnb_pricing['realSum'] - min_realSum
    ) / (denominator)

# Calculate score, handling potential division by zero (realSum_norm could be 0)
df_airbnb_pricing['score'] = (
    1 / df_airbnb_pricing['realSum_norm'].replace(0, pd.NA).fillna(df_airbnb_pricing['realSum_norm'].mean()) # Replace 0 with NA, then fill with mean
) * df_airbnb_pricing['guest_satisfaction_overall'] * df_airbnb_pricing['attr_index_norm']

# Replace infinite values in 'score' with NaN, then fill with the mean of the column
df_airbnb_pricing['score'] = df_airbnb_pricing['score'].replace([float('inf'), -float('inf')], pd.NA)
df_airbnb_pricing['score'] = df_airbnb_pricing['score'].fillna(df_airbnb_pricing['score'].mean())

airbnb_silver_layer = df_airbnb_pricing.copy() # Assign cleaned pricing data to silver layer

print("Silver Layer shape (without events):", airbnb_silver_layer.shape)

# Final check for any null values and drop if any exist
initial_rows = airbnb_silver_layer.shape[0]
airbnb_silver_layer.dropna(inplace=True)
if airbnb_silver_layer.shape[0] < initial_rows:
    print(f"Dropped {initial_rows - airbnb_silver_layer.shape[0]} rows due to missing values.")

print(f"Final Silver Layer shape (after null handling): {airbnb_silver_layer.shape}")
print(f"Number of nulls remaining in silver layer: {airbnb_silver_layer.isnull().sum().sum()}")

# ----------------------
# Preview the cleaned & enriched dataset
# ----------------------
print("✅ Preview of the Silver Layer (first 10 rows):")
display(airbnb_silver_layer.head(10))

print("\nRandom sample of 5 rows:")
display(airbnb_silver_layer.sample(5))

In [ ]:
average_realSum_per_city = airbnb_silver_layer.groupby('city')['realSum'].mean().reset_index()
average_realSum_per_city = average_realSum_per_city.sort_values(by='realSum', ascending=False)
print("Average 'realSum' per city:")
display(average_realSum_per_city)

In [ ]:
pivot_df = average_realSum_per_city_day_type.pivot_table(index='city', columns='day_type', values='realSum').reset_index()
# Rename columns for clarity
pivot_df.columns.name = None
pivot_df = pivot_df.rename(columns={'Weekdays': 'average_realSum_weekdays', 'Weekends': 'average_realSum_weekends'})

print("Average 'realSum' per city, with separate columns for weekdays and weekends:")
display(pivot_df)

In [ ]:
# Calculate min realSum per city and day type
min_realSum_per_city_day_type = airbnb_silver_layer.groupby(['city', 'day_type'])['realSum'].min().reset_index()
pivot_min_df = min_realSum_per_city_day_type.pivot_table(index='city', columns='day_type', values='realSum').reset_index()
pivot_min_df.columns.name = None
pivot_min_df = pivot_min_df.rename(columns={'Weekdays': 'min_realSum_weekdays', 'Weekends': 'min_realSum_weekends'})

print("Minimum 'realSum' per city, with separate columns for weekdays and weekends:")
display(pivot_min_df)

# Calculate max realSum per city and day type
max_realSum_per_city_day_type = airbnb_silver_layer.groupby(['city', 'day_type'])['realSum'].max().reset_index()
pivot_max_df = max_realSum_per_city_day_type.pivot_table(index='city', columns='day_type', values='realSum').reset_index()
pivot_max_df.columns.name = None
pivot_max_df = pivot_max_df.rename(columns={'Weekdays': 'max_realSum_weekdays', 'Weekends': 'max_realSum_weekends'})

print("\nMaximum 'realSum' per city, with separate columns for weekdays and weekends:")
display(pivot_max_df)

In [ ]:
# ----------------------
# Save as Parquet locally and upload to GCS
# ----------------------
silver_local_path = "airbnb_enriched_silver.parquet"
airbnb_silver_layer.to_parquet(silver_local_path, index=False)

blob = bucket.blob(SILVER_PARQUET_PATH)
blob.upload_from_filename(silver_local_path)

print("✅ Silver Layer saved to GCS in Parquet format at:", f"gs://{BUCKET_NAME}/{SILVER_PARQUET_PATH}")

In [ ]:
airbnb_silver_layer.info()

In [ ]:
# 0. Install dependencies
!pip install pandas pyarrow
!pip install google-cloud-storage
!pip install tableauhyperapi

import pandas as pd
from google.colab import auth
from tableauhyperapi import HyperProcess, Connection, TableDefinition, SqlType, TableName, Telemetry, Inserter, CreateMode
from io import BytesIO

# ----------------------
# 1. Authenticate GCP
# ----------------------
auth.authenticate_user()
client = storage.Client()
bucket = client.bucket("dw-airbnb-data-lake")

# ----------------------
# 2. Load Airbnb Silver Parquet from GCS
# ----------------------
parquet_path = "silver_layer/pricing_table/airbnb_pricing_data.parquet"
blob = bucket.blob(parquet_path)
df_airbnb = pd.read_parquet(BytesIO(blob.download_as_bytes()))

print("✅ Airbnb pricing data loaded from GCS")
print("Shape:", df_airbnb.shape)
display(df_airbnb.head(10))

# ----------------------
# 3. Optional: Keep only relevant columns for Tableau
# ----------------------
cols_to_keep = ['city','realSum','score','guest_satisfaction_overall',
                'attr_index_norm', 'day_type', 'room_type', 'person_capacity', 'host_is_superhost']

df_airbnb_tableau = df_airbnb[cols_to_keep]
display(df_airbnb_tableau.head(10))

# ----------------------
# 4. Export to .hyper for Tableau Cloud
# ----------------------
hyper_file = "airbnb_pricing_data.hyper"

# Helper function to map pandas dtypes to Hyper API SqlTypes
def map_pandas_to_hyper_type(pandas_dtype):
    if pd.api.types.is_integer_dtype(pandas_dtype):
        return SqlType.big_int()
    elif pd.api.types.is_float_dtype(pandas_dtype):
        return SqlType.double()
    elif pd.api.types.is_bool_dtype(pandas_dtype):
        return SqlType.bool()
    elif pd.api.types.is_datetime64_any_dtype(pandas_dtype):
        return SqlType.timestamp()
    else:
        return SqlType.text() # Default to text for other types like object (strings)

# Create column definitions for Tableau
tableau_columns_defs = []
for col_name in df_airbnb_tableau.columns:
    pandas_dtype = df_airbnb_tableau[col_name].dtype
    hyper_type = map_pandas_to_hyper_type(pandas_dtype)
    tableau_columns_defs.append(TableDefinition.Column(col_name, hyper_type))

with HyperProcess(telemetry=Telemetry.SEND_USAGE_DATA_TO_TABLEAU) as hyper:
    with Connection(endpoint=hyper.endpoint, database=hyper_file, create_mode=CreateMode.CREATE_AND_REPLACE) as conn:
        # Define table schema using the generated column definitions
        table = TableDefinition(
            table_name=TableName("airbnb_pricing_data"), # Removed 'Extract' schema name
            columns=tableau_columns_defs
        )
        conn.catalog.create_table(table)

        # Insert the data
        with Inserter(conn, table) as inserter:
            inserter.add_rows(df_airbnb_tableau.values.tolist())
            inserter.execute()

# ----------------------
# 5. Download the .hyper file for Tableau Cloud
# ----------------------
from google.colab import files
files.download(hyper_file)
print("✅ .hyper file ready for Tableau Cloud upload")

In [ ]:
try:
    display(airbnb_silver_layer.describe())
except NameError:
    print("The 'airbnb_silver_layer' DataFrame is not defined. Please ensure you have run the preceding cells that load and process the Airbnb pricing data.")